# Imports 

In [38]:
import pandas as pd
import numpy as np
import os

import seaborn

## repeated printouts
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## Read in data and make sure relevant columns are string/character 

- San Diego data: `naics_code` and `account_key`
- NAICS details data: `naics` (North American Industry Classification Systems)

Run code below; if pulling from github, pathname should be fine; if working elsewhere may need to edit path name at read in 

In [39]:
# uncomment if u need to find your path: os.getcwd()
SD = sd_df = pd.read_csv("../public_data/sd_df.csv")
NA = naics_df = pd.read_csv("../public_data/naics_df.csv")

In [40]:
cols_sd_use = ["naics_code", "account_key"]
cols_naics_use = ["naics"]

sd_df[cols_sd_use] = sd_df[cols_sd_use].astype(str)
naics_df[cols_naics_use] = naics_df[cols_naics_use].astype(str)

sd_df.dtypes
naics_df.dtypes
sd_df.head()
naics_df.head()

account_key          object
dba_name             object
council_district     object
naics_code           object
naics_description    object
naics_nchar           int64
dtype: object

naics                object
naics_description    object
dtype: object

,account_key,dba_name,council_district,naics_code,naics_description,naics_nchar
0,1974000448,ERNST & YOUNG LLP,cd_1,541211,OFFICES OF CERTIFIED PUBLIC ACCOUNTANTS,6
1,1974011093,HECHT SOLBERG ROBINSON GOLDBERG & BAGLEY LLP,cd_3,5411,LEGAL SERVICES,4
2,1978039819,RSM US LLP,cd_1,541211,OFFICES OF CERTIFIED PUBLIC ACCOUNTANTS,6
3,1978042092,THORSNES BARTOLOTTA MCGUIRE LLP,cd_3,5411,LEGAL SERVICES,4
4,1979046817,KORENIC & WOJDOWSKI LLP,cd_7,5412,ACCOUNTING/TAX PREP/BOOKKEEP/PAYROLL SERVICES,4


,naics,naics_description
0,111140,Wheat Farming
1,111160,Rice Farming
2,111150,Corn Farming
3,111110,Soybean Farming
4,111120,Oilseed (except Soybean) Farming


## "Inner join"- retain only San Diego businesses with details on their NAICS code

- Use the `naics_code` column in the San Diego business data as the join key
- Use the `naics` column in the NAICS code details data as the join key

- Do an inner join of the San Diego data onto the NAICS code details using these join keys
- After the inner join, print some examples of San Diego businesses lost in the merge
- Use value_counts() on the `naics_nchar` column in the San Diego data to see why they might have gotten lost


In [45]:
# your code here

sd_naics_inner = pd.merge(sd_df, naics_df, left_on = "naics_code",
                          right_on = "naics", how = "inner")

sd_df.shape
sd_naics_inner.shape
print(f"Retained {sd_naics_inner.shape[0] / sd_df.shape[0]:.4f} of the original SD rows")

lost_keys = set(sd_df.account_key) - set(sd_naics_inner.account_key)
sd_lost = sd_df[sd_df.account_key.isin(lost_keys)]
sd_lost[["account_key", "naics_code", "naics_nchar"]].head(10)

sd_lost.naics_nchar.value_counts()
sd_df.naics_nchar.value_counts()
naics_df.naics.str.len().value_counts()

(353, 7)

(52, 9)

Retained 0.1473 of the original SD rows


,account_key,naics_code,naics_nchar
1,1974011093,5411,4
3,1978042092,5411,4
4,1979046817,5412,4
5,1979053082,5411,4
6,1981000840,5411,4
10,1987006330,5411,4
11,1987007978,5411,4
12,1988000382,7221,4
13,1989013971,48841,5
14,1990012245,54111,5


naics_nchar
5    150
4     89
3     42
2     23
6     14
Name: count, dtype: int64

naics_nchar
5    150
4     89
6     49
3     42
2     23
Name: count, dtype: int64

naics
6    2270
Name: count, dtype: int64


1. **Perform inner merge**: `pd.merge(sd_df, naics_df, left_on="naics_code", right_on="naics", how="inner")` - Merges two DataFrames using an inner join, matching `naics_code` column from `sd_df` with `naics` column from `naics_df`. Only keeps rows where matches exist in both DataFrames.

2. **Check original and merged shapes**: `sd_df.shape` and `sd_naics_inner.shape` - Display the dimensions (rows, columns) of both the original and merged DataFrames.

3. **Calculate retention rate**: `print(f"Retained {sd_naics_inner.shape[0] / sd_df.shape[0]:.4f} of the original SD rows")` - Calculates and prints what fraction of original rows were retained after the merge, formatted to 4 decimal places.

4. **Find lost account keys**: `set(sd_df.account_key) - set(sd_naics_inner.account_key)` - Uses set difference to identify which account keys from the original DataFrame were lost during the merge.

5. **Filter lost rows**: `sd_df[sd_df.account_key.isin(lost_keys)]` - Creates a DataFrame containing only the rows that were lost during the merge process.

6. **Display sample of lost data**: `sd_lost[["account_key", "naics_code", "naics_nchar"]].head(10)` - Shows the first 10 rows of specific columns from the lost data to examine what was dropped.

7. **Analyze data characteristics**: The last three lines use `value_counts()` to examine the distribution of NAICS code lengths in the lost data, original data, and NAICS reference DataFrame respectively.

## "Left join"- retain all sd businesses even if naics code isn't in `naics_df`

- Using the same join keys as above, and treating the San Diego businesses as the left hand side data, left join the naics code details onto the San Diego businesses
- Use the `indicator` argument within merge to create an indicator, `naics_merge_status`, to help with later merge diagnostics and examine sample of ones that didn't merge
- Use the `suffixes` argument within merge to add `_sd` as the left suffix, `_census` as the right suffix
- Use `naics_merge_status` in the merged result to look at a sample of San Diego businesses that weren't matched with `naics_df`

In [46]:
# your code here

sd_naics_left = pd.merge(sd_df, naics_df, left_on = "naics_code",
                         right_on = "naics", how = "left",
                         indicator = "naics_merge_status", suffixes = ["_sd", "_census"])

sd_df.shape
sd_naics_left.shape
sd_naics_left.naics_merge_status.value_counts()
sd_naics_left.naics_merge_status.value_counts(normalize = True)

sd_naics_left[sd_naics_left.naics_merge_status == "left_only"][["account_key", "naics_code", "naics_nchar"]].head(10)

(353, 7)

(370, 10)

naics_merge_status
left_only     318
both           52
right_only      0
Name: count, dtype: int64

naics_merge_status
left_only     0.859459
both          0.140541
right_only    0.000000
Name: proportion, dtype: float64

,account_key,naics_code,naics_nchar
1,1974011093,5411,4
3,1978042092,5411,4
4,1979046817,5412,4
5,1979053082,5411,4
6,1981000840,5411,4
10,1987006330,5411,4
11,1987007978,5411,4
12,1988000382,7221,4
13,1989013971,48841,5
14,1990012245,54111,5



1. `pd.merge()` - Calls pandas merge function to join two DataFrames together

2. `sd_df, naics_df` - The two DataFrames being merged (left and right DataFrames respectively)

3. `left_on = "naics_code", right_on = "naics"` - Specifies the column names to join on; "naics_code" from the left DataFrame matches with "naics" from the right DataFrame

4. `how = "left"` - Performs a left join, keeping all rows from the left DataFrame (sd_df) and matching rows from the right DataFrame (naics_df)

5. `indicator = "naics_merge_status"` - Adds a new column that shows the source of each row ("left_only", "right_only", or "both")

6. `suffixes = ["_sd", "_census"]` - When column names overlap between DataFrames, adds these suffixes to distinguish them

7. The result is stored in `sd_naics_left`, which contains all records from sd_df with matching naics information added where available.

## Use group by and agg to see if there are differences in merge rates by area

- Using the left-joined dataframe created in previous step, create a boolean indicator, `is_lost`, that equals `True` if the merge indicator is equal to "left_only" (and is otherwise false)
- Find the proportion of businesses by council district that were lost in the left join (SD businesses that failed to match to `naics_df`). To do this, group by `council_district` and use the shortcut of taking the mean of the `is_lost` indicator to find the proportions lost by `council_district`. 

In [43]:
# your code here

sd_naics_left["is_lost"] = sd_naics_left.naics_merge_status == "left_only"
sd_naics_left.is_lost.value_counts()

lost_by_district = sd_naics_left.groupby("council_district").agg(prop_lost = ("is_lost", "mean"),
                                                                n_businesses = ("is_lost", "size")).reset_index()
lost_by_district.sort_values(by = "prop_lost", ascending = False)

is_lost
True     318
False     52
Name: count, dtype: int64

,council_district,prop_lost,n_businesses
3,cd_4,1.000000,4
8,cd_9,1.000000,8
7,cd_8,0.933333,15
6,cd_7,0.918919,37
4,cd_5,0.916667,24
0,cd_1,0.880597,67
5,cd_6,0.859649,57
2,cd_3,0.812500,112
1,cd_2,0.804348,46



1. `sd_naics_left["is_lost"] = sd_naics_left.naics_merge_status == "left_only"` - This creates a new boolean column called "is_lost" in the dataframe. It sets the value to `True` where the "naics_merge_status" column equals "left_only", and `False` otherwise.

2. `sd_naics_left.is_lost.value_counts()` - This counts the number of `True` and `False` values in the newly created "is_lost" column, showing how many businesses are classified as "lost" versus not lost.

3. `lost_by_district = sd_naics_left.groupby("council_district").agg(...)` - This groups the data by "council_district" and performs aggregation operations on each group.

4. `prop_lost = ("is_lost", "mean")` - This calculates the proportion (mean) of lost businesses for each district, since the mean of boolean values gives the proportion of `True` values.

5. `n_businesses = ("is_lost", "size")` - This counts the total number of businesses in each district using the size of each group.

6. `.reset_index()` - This converts the grouped result back to a regular dataframe with "council_district" as a regular column.

7. `lost_by_district.sort_values(by = "prop_lost", ascending = False)` - This sorts the results by the "prop_lost" column in descending order, showing districts with the highest proportion of lost businesses first.

## Optional challenge exercise: add lagging 0's and see if merge rate improves

You noticed earlier that a big reason for non-matches is that the San Diego tax certificate NAICS codes were often less than six digits long, while the Census ones were always 6 digits.

You wonder if this is an issue where 0's in some of the SD's data naics codes got cut off (eg 540000 became 54), and if so whether adding these lagging zeros would improve the merge rate in a left join.

- Pad the `naics_code` column in `sd_df` with 0's to get that column up to 6-digits, using one of two approaches: 
  1. `str.pad` in pandas (https://pandas.pydata.org/docs/reference/api/pandas.Series.str.pad.html)
  2. for more of a challenge, write your own function! It should check the # of digits in the naics code string and pad it with 0's at the end up to 6 digits. To execute your function, use row-wise apply: `df.apply(lambda row: funcname(row.column), axis=1)`.
- Perform the same left join as above and see how the match rate changes
- Create an indicator variable--`is_new_match`---for new matches under the padded NAICS code; compare the `naics_description` column from San Diego versus Census in the new dataset for a sample of these new matches and comment on whether the padding seems to be correct

In [44]:
sd_df["naics_code_padded"] = sd_df.naics_code.str.pad(width = 6, side = "right", fillchar = "0")
sd_df[["naics_code", "naics_code_padded"]].head(10)

sd_naics_left_pad = pd.merge(sd_df, naics_df, left_on = "naics_code_padded",
                             right_on = "naics", how = "left",
                             indicator = "naics_merge_status", suffixes = ["_sd", "_census"])

sd_naics_left.naics_merge_status.value_counts(normalize = True)
sd_naics_left_pad.naics_merge_status.value_counts(normalize = True)

old_matched = set(sd_naics_left[sd_naics_left.naics_merge_status == "both"].account_key)
sd_naics_left_pad["is_new_match"] = ((sd_naics_left_pad.naics_merge_status == "both") &
                                     (~sd_naics_left_pad.account_key.isin(old_matched)))
sd_naics_left_pad.is_new_match.value_counts()

sd_naics_left_pad[sd_naics_left_pad.is_new_match][["naics_code", "naics_code_padded",
                                                   "naics_description_sd", "naics_description_census"]].sample(10, random_state = 422)

,naics_code,naics_code_padded
0,541211,541211
1,5411,541100
2,541211,541211
3,5411,541100
4,5412,541200
5,5411,541100
6,5411,541100
7,541211,541211
8,541211,541211
9,541211,541211


naics_merge_status
left_only     0.859459
both          0.140541
right_only    0.000000
Name: proportion, dtype: float64

naics_merge_status
both          0.53139
left_only     0.46861
right_only    0.00000
Name: proportion, dtype: float64

is_new_match
False    261
True     185
Name: count, dtype: int64

,naics_code,naics_code_padded,naics_description_sd,naics_description_census
55,53311,533110,LESSORS OF OTHER NONFINANCIAL INTANGIBLE ASSE,Lessors of Nonfinancial Intangible Assets (exc...
206,53111,531110,LESSORS OF RESIDENTIAL BUILDINGS & DWELLINGS,Lessors of Residential Buildings and Dwellings
223,53111,531110,LESSORS OF RESIDENTIAL BUILDINGS & DWELLINGS,Lessors of Residential Buildings and Dwellings
337,45439,454390,OTHER DIRECT SELLING ESTABLISHMENTS,Other Direct Selling Establishments
242,31213,312130,WINERIES,Wineries
67,54111,541110,OFFICES OF LAWYERS,Offices of Lawyers
366,54133,541330,ENGINEERING SERVICES,Engineering Services
336,45439,454390,OTHER DIRECT SELLING ESTABLISHMENTS,Other Direct Selling Establishments
76,53111,531110,LESSORS OF RESIDENTIAL BUILDINGS & DWELLINGS,Lessors of Residential Buildings and Dwellings
204,31192,311920,COFFEE & TEA MFG,Coffee and Tea Manufacturing


# Assistant
Here's a step-by-step explanation of the selected code:

1. **Merge DataFrames**: `pd.merge()` performs a left join between `sd_df` and `naics_df`, matching records where the padded NAICS code from `sd_df` equals the NAICS code in `naics_df`. The `indicator` parameter tracks merge status, and `suffixes` distinguish columns with the same name.

2. **Compare merge results**: Two `value_counts(normalize=True)` calls show the proportion of records that matched in both the original merge (`sd_naics_left`) and the new padded merge (`sd_naics_left_pad`).

3. **Identify previously matched records**: Creates a set of `account_key` values that were successfully matched in the original merge (where `naics_merge_status == "both"`).

4. **Flag new matches**: Creates a boolean column `is_new_match` that identifies records that matched in the padded merge but didn't match in the original merge (records that are in "both" status but not in the `old_matched` set).

5. **Count new matches**: Shows how many records are newly matched due to the padding.

6. **Sample new matches**: Displays 10 random examples of newly matched records, showing the original NAICS code, padded version, and descriptions from both datasets to verify the matching worked correctly.


1. `sd_df["naics_code_padded"] = sd_df.naics_code.str.pad(width = 6, side = "right", fillchar = "0")` - This creates a new column called "naics_code_padded" by taking the existing "naics_code" column and padding it to ensure all codes are exactly 6 characters long.

2. `width = 6` - Sets the target width to 6 characters total.

3. `side = "right"` - Specifies that padding should be added to the right side of the string.

4. `fillchar = "0"` - Uses "0" as the character to pad with when the original code is shorter than 6 characters.

5. `sd_df[["naics_code", "naics_code_padded"]].head(10)` - Displays the first 10 rows showing both the original "naics_code" and the new "naics_code_padded" columns for comparison.

This padding ensures that NAICS codes have a consistent format, which is important for accurate matching operations later in the code.

Comment on the padding: If the padded codes are correct, the `naics_description_sd` and `naics_description_census` columns for the new matches should describe the same broad industry (the first 2-3 digits carry the sector, so padding a truncated code with trailing 0's lands on the sector-level Census code). Where the two descriptions disagree, the original code was likely a genuinely different 6-digit code that lost leading rather than trailing information, so the padded match is spurious.